In [1]:
import torch
import os
from transformers import AutoModel, AutoTokenizer, AutoProcessor, SiglipModel
from transformers import SiglipImageProcessor, SiglipModel

MODEL_DIR = '/mnt/hdd/checkpoints/'
_MODELS = {
    "siglip2-ViT-B-16": os.path.join(MODEL_DIR, 'siglip2')
}

# Needs to have a 'tokenizer' attribute that is accessed
class SigLIPWrapper:
    def __init__(self, model_name, device):
        self.model = SiglipModel.from_pretrained(_MODELS[model_name])
        self.processor = SiglipImageProcessor.from_pretrained(_MODELS[model_name])
        # inputs = tokenizer(["a photo of a cat", "a photo of a dog"], padding="max_length", return_tensors="pt")
        self.tokenizer = AutoTokenizer.from_pretrained(_MODELS[model_name])
        self.device = device

    @torch.no_grad()
    def encode_image(self, inputs): # expects tensor
        with torch.no_grad():
            image_embeddings = self.model.get_image_features(**inputs)
        return image_embeddings

    @torch.no_grad()
    def encode_text(self, inputs): # expects tensor
        with torch.no_grad():
            text_features = self.model.get_text_features(**inputs)
        return text_features

def load_siglip(name, device = "cuda" if torch.cuda.is_available() else "cpu"):
    if name not in _MODELS:
        raise RuntimeError(f"Model {name} not found in available models: {list(_MODELS.keys())}")

    wrapper = SigLIPWrapper(name, device)
    return wrapper, wrapper.processor

In [2]:
model, processor = load_siglip('siglip2-ViT-B-16')

In [3]:
from PIL import Image
img=Image.open('/mnt/hdd/datasets/CoOp/caltech-101/101_ObjectCategories/accordion/image_0001.jpg')
inputs = processor(images=img, return_tensors="pt")
model.encode_image(inputs).shape

torch.Size([1, 768])

In [4]:
candidate_labels = ["a Pallas cat", "a lion", "a Siberian tiger"]
inputs = model.tokenizer(candidate_labels, padding="max_length", max_length=64, return_tensors="pt")
model.encode_text(inputs).shape

torch.Size([3, 768])